# Elastic-Net Regression Analysis (PFC, mFC dataset)

Port of `code/LEC_elasticnet_regression.ipynb` to the mPFC dataset under
`mFC_data/data/`. Same elastic-net pipeline (per-neuron regression of normalized
firing rate onto location × goal-progress × lag anchors, El-Gaby et al. 2024,
via [`elasticnet_regression_v2.py`](elasticnet_regression_v2.py)). Only the
data-loading cells are PFC-specific — the PFC loader builds the LEC-shaped
`data_dic` from disk and (with `compute_norm=True`) also populates the
`Neurons_norm` / `Locs_norm` fields the elastic-net consumes.

In [1]:
import numpy as np
import scipy.stats as st
from scipy import stats
from scipy.stats import zscore
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os, pickle
from tqdm import tqdm

In [2]:
DATA_FOLDER = '/ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/data'
META = os.path.join(DATA_FOLDER, 'MetaData')

In [3]:
# Load the canonical recday list (25 double-day recordings, ABCD tasks only)
mouse_recdays = list(np.load(os.path.join(META, 'combined_ABCDonly_days.npy')).astype(str))
print(f'{len(mouse_recdays)} recdays in combined_ABCDonly_days.npy')
print('first 3:', mouse_recdays[:3])

25 recdays in combined_ABCDonly_days.npy
first 3: [np.str_('ab03_01092023_02092023'), np.str_('ab03_05092023_06092023'), np.str_('ab03_29082023_30082023')]


In [ ]:
# Build the data_dic from the PFC directory layout with normalized fields populated.
# compute_norm=True adds Neurons_norm (n_neurons, n_full_trials, 360) and Locs_norm
# (n_full_trials, 360) per session — required by the elastic-net pipeline.
# Memory note: this is ~4 GB across all 25 recdays. For a quick first pass, uncomment
# the next line to slice down.
# mouse_recdays = mouse_recdays[:5]

from glm_analysis_v2 import build_data_dic_from_pfc
data_dic = build_data_dic_from_pfc(DATA_FOLDER, mouse_recdays, compute_norm=True)
mouse_recdays = sorted(data_dic.keys())   # in case any were skipped
print(f'\n{len(mouse_recdays)} recdays loaded with Neurons_norm + Locs_norm')

  ab03_01092023_02092023: 7 sessions
  ab03_05092023_06092023: 8 sessions
  ab03_29082023_30082023: 9 sessions
  ah03_12082021_13082021: 8 sessions
  ah03_18082021_19082021: 8 sessions
  ah04_01122021_02122021: 8 sessions
  ah04_05122021_06122021: 8 sessions


In [ ]:
# Build valid_sessions_dic: for each mouse_recday, keep one session per unique task structure.
# Identical logic to the LEC notebook; works on PFC because num_trials and Task are present.
valid_sessions_dic = {}
for mouse_recday in mouse_recdays:
    valid_sessions = []
    tasks = []
    for session in list(data_dic[mouse_recday].keys()):
        if session == 'valid_sessions':
            continue
        if data_dic[mouse_recday][session]['num_trials'] < 5:
            print(f'{mouse_recday} session {session}: not enough trials, skipping')
            continue
        if 'defaultdict' in str(data_dic[mouse_recday][session]['Task']):
            print(f'{mouse_recday} session {session}: no task, skipping')
            continue
        if not any(np.array_equal(data_dic[mouse_recday][session]['Task'], candidate)
                   for candidate in tasks):
            tasks.append(data_dic[mouse_recday][session]['Task'])
            valid_sessions.append(session)
    print(f'{mouse_recday}: {valid_sessions}')
    valid_sessions_dic[mouse_recday] = valid_sessions

## Configure + run regression

In [ ]:
# Load the elastic net regression module
exec(open('elasticnet_regression_v2.py').read())

# Configure for your 22-location maze
config = RegressionConfig(
    num_locations=9,
    num_goal_progress_bins=3,
    num_task_states=4,
    num_lags=12,
    alpha=0.01
)

# Run for one recording
mouse_recday = list(data_dic.keys())[5]
results = run_cross_validated_regression(
    data_dic, mouse_recday, config,
    valid_sessions=valid_sessions_dic[mouse_recday]
)

print(f"Mean prediction correlation: {np.nanmean(results['mean_correlations']):.3f}")

plot_prediction_correlations(results, mouse_recday=list(data_dic.keys())[5])




### El-Gaby filtered prediction correlations

Re-runs the all-lag vs non-zero-coef prediction comparison, but restricted to the **El-Gaby non-zero-lag cohort** (neurons whose top-3 highest regression coefficients are all at lag 2..num_lags-2). The figure above shows the *same* 81 neurons in both panels (just with lag-0 coefs zeroed in panel 2); this cell answers the different question: how well do we predict the subset of neurons the paper calls *non-zero-lag neurons*?

In [ ]:
# Apples-to-apples version of plot_prediction_correlations, filtered to the
# El-Gaby "non-zero lag" cohort: neurons whose top-3 betas are all at lag 2..num_lags-2.
# Uses identify_nonzero_lag_neurons() from elasticnet_regression_v2.py.

nz_mask, _ = identify_nonzero_lag_neurons(results, config)
mean_corrs_all = results['mean_correlations']
mean_corrs_nz_pred = results['mean_correlations_nonzero']

valid_all = nz_mask & ~np.isnan(mean_corrs_all)
valid_nz  = nz_mask & ~np.isnan(mean_corrs_nz_pred)

corrs_all = mean_corrs_all[valid_all]
corrs_nz  = mean_corrs_nz_pred[valid_nz]

print(f"El-Gaby non-zero-lag neurons: {nz_mask.sum()} of "
      f"{(~np.isnan(mean_corrs_all)).sum()} state-tuned")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: all-lag prediction, El-Gaby cohort only
ax = axes[0]
ax.hist(corrs_all, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
ax.axvline(0, color='black', linestyle='--', linewidth=2)
m1 = np.mean(corrs_all)
ax.axvline(m1, color='red', linewidth=2, label=f'Mean = {m1:.3f}')
if len(corrs_all) > 1:
    t1, p1 = stats.ttest_1samp(corrs_all, 0)
    ax.text(0.05, 0.95, f'n={len(corrs_all)}\nt={t1:.2f}\np={p1:.2e}',
            transform=ax.transAxes, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('Correlation (predicted vs actual)')
ax.set_ylabel('Number of neurons')
ax.set_title(f'All-lag prediction\n(El-Gaby top-3 neurons, n={len(corrs_all)})')
ax.legend()

# Panel 2: lag-0-zeroed prediction, El-Gaby cohort only
ax = axes[1]
ax.hist(corrs_nz, bins=30, color='darkorange', alpha=0.7, edgecolor='black')
ax.axvline(0, color='black', linestyle='--', linewidth=2)
m2 = np.mean(corrs_nz)
ax.axvline(m2, color='red', linewidth=2, label=f'Mean = {m2:.3f}')
if len(corrs_nz) > 1:
    t2, p2 = stats.ttest_1samp(corrs_nz, 0)
    ax.text(0.05, 0.95, f'n={len(corrs_nz)}\nt={t2:.2f}\np={p2:.2e}',
            transform=ax.transAxes, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('Correlation (lag-0 coefficients zeroed)')
ax.set_ylabel('Number of neurons')
ax.set_title(f'Non-zero-coef prediction\n(El-Gaby top-3 neurons, n={len(corrs_nz)})')
ax.legend()

# Panel 3: all-lag vs lag-0-zeroed scatter, El-Gaby cohort
ax = axes[2]
both = nz_mask & ~np.isnan(mean_corrs_all) & ~np.isnan(mean_corrs_nz_pred)
ax.scatter(mean_corrs_all[both], mean_corrs_nz_pred[both], alpha=0.6, s=25, c='teal')
ax.plot([-1, 1], [-1, 1], 'k--', linewidth=1)
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_xlabel('All-lag correlation')
ax.set_ylabel('Non-zero-coef correlation')
ax.set_title(f'All-lag vs Non-zero-coef\n(El-Gaby top-3, n={both.sum()})')
ax.set_aspect('equal')

plt.suptitle(
    'Prediction correlations restricted to El-Gaby non-zero-lag neurons '
    '(top-3 betas at lag 2 to num_lags-2)',
    fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:

fig, nzl_mask = plot_nonzero_lag_neuron_correlations(results, config, min_lag_distance=2, mouse_recday=list(data_dic.keys())[5])
# Plot example beta matrices for individual neurons (top 6 by correlation)
plot_example_betas(results, config, num_examples=20)

# Plot population summary of betas
# plot_population_beta_summary(results, config)

In [ ]:
# Plot example beta matrices for individual neurons (top 6 by correlation)
plot_example_betas(results, config, num_examples=6)

# Plot population summary of betas
plot_population_beta_summary(results, config)

## Tuning-curve cross-validated regression

In [ ]:
mouse_recday = list(data_dic.keys())[5]
mouse_recday = list(data_dic.keys())[6]
# Run tuning curve version
results_tc = run_cross_validated_regression_tuning_curves(
    data_dic, mouse_recday, config, 
    valid_sessions=valid_sessions_dic[mouse_recday]
)

# Polar plots of top neurons - ALL neurons
fig_tc, selected_neurons = plot_polar_tuning_curves(results_tc, config, num_examples=6, 
                         sort_by='correlation', mouse_recday=mouse_recday)

# Beta matrices for those same neurons
plot_example_betas(results_tc, config, neuron_indices=selected_neurons)

# Polar plots of ONLY non-zero lag neurons (peaks at lag >= 2 away from 'now')
fig_tc_nz, selected_neurons_nz = plot_polar_tuning_curves(results_tc, config, num_examples=6, 
                         sort_by='correlation', mouse_recday=mouse_recday,
                         nonzero_lag_only=True)

# Beta matrices for those same non-zero lag neurons
plot_example_betas(results_tc, config, neuron_indices=selected_neurons_nz)

# Tuning correlation distribution for all
# plot_tuning_correlation_distribution(results_tc, mouse_recday=mouse_recday)

plot_tuning_correlation_distribution(results_tc, config, mouse_recday=mouse_recday)



### 4-state-mean correlation (goal-progress confound-controlled)

The 360-bin tuning correlation above mixes two effects: state-level tuning (which state fires highest?) and within-state goal-progress structure (ramps inside each 90-bin window). Because most LEC neurons have strong within-state ramps in both predicted and actual, that shared structure can **inflate** the 360-bin correlation even when the model isn't capturing state-level specificity.

The cell below collapses each 360-bin tuning curve to a **4-vector** (the mean over each 90-bin state window) and correlates those instead. This isolates state-level tuning by removing the within-state ramp as a shared confound. If 4-state r is systematically lower than 360-bin r, the original metric was being inflated by ramp structure; if comparable, the model is genuinely capturing state-specificity.

Per-neuron p-values on the n=4 correlation are noisy by construction — the inference is at the population level (mean r + one-sample t-test against 0).

In [ ]:
# 360-bin tuning corr vs 4-state-mean tuning corr.
# Both metrics come from results_tc (added by run_cross_validated_regression_tuning_curves in v2).

mean_360 = results_tc['mean_tuning_correlations']
mean_state = results_tc['mean_state_correlations']

valid = ~np.isnan(mean_360) & ~np.isnan(mean_state)
r_360 = mean_360[valid]
r_state = mean_state[valid]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: 360-bin distribution
ax = axes[0]
ax.hist(r_360, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
ax.axvline(0, color='black', linestyle='--', linewidth=2)
m = np.mean(r_360)
ax.axvline(m, color='red', linewidth=2, label=f'Mean = {m:.3f}')
t, p = stats.ttest_1samp(r_360, 0)
ax.text(0.05, 0.95, f'n={len(r_360)}\nt={t:.2f}\np={p:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('Tuning correlation (360 bins)')
ax.set_ylabel('Number of neurons')
ax.set_title('360-bin tuning corr\n(mixes state + within-state)')
ax.legend()

# Panel 2: 4-state distribution
ax = axes[1]
ax.hist(r_state, bins=30, color='darkorange', alpha=0.7, edgecolor='black')
ax.axvline(0, color='black', linestyle='--', linewidth=2)
m_state = np.mean(r_state)
ax.axvline(m_state, color='red', linewidth=2, label=f'Mean = {m_state:.3f}')
t_st, p_st = stats.ttest_1samp(r_state, 0)
ax.text(0.05, 0.95, f'n={len(r_state)}\nt={t_st:.2f}\np={p_st:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('State-mean correlation (4 states)')
ax.set_ylabel('Number of neurons')
ax.set_title('4-state tuning corr\n(goal-progress confound-controlled)')
ax.legend()

# Panel 3: per-neuron scatter
ax = axes[2]
ax.scatter(r_360, r_state, alpha=0.5, s=20, c='teal')
ax.plot([-1, 1], [-1, 1], 'k--', linewidth=1)
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_aspect('equal')
ax.set_xlabel('360-bin tuning corr')
ax.set_ylabel('4-state tuning corr')
ax.set_title('Per-neuron: 360-bin vs 4-state')

plt.suptitle(f'Goal-progress confound check: {mouse_recday}',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

#### Aggregator: mean vs max per state window

The 4-state correlation above used **mean** per 90-bin state window. Below we recompute the same per-neuron correlation using **max** instead — testing whether the predicted-vs-actual alignment is driven by peak responses rather than sustained drive. Recomputed from `results_tc['cv_actual_tuning']` and `['cv_predicted_tuning']` so no re-fitting is needed.

In [ ]:
# 4-state correlation using MAX per state window (alternative to MEAN above).
# Recomputed from the stored full 360-bin tuning curves; no re-fitting needed.

cv_actual = results_tc['cv_actual_tuning']          # (n_neurons, n_sessions, 360)
cv_pred   = results_tc['cv_predicted_tuning']
n_neurons, n_sessions, _ = cv_actual.shape
n_states  = config.num_task_states
bps       = config.num_bins_per_state

actual_max = np.nanmax(cv_actual.reshape(n_neurons, n_sessions, n_states, bps), axis=3)
pred_max   = np.nanmax(cv_pred.reshape(n_neurons, n_sessions, n_states, bps), axis=3)

cv_state_corrs_max = np.full((n_neurons, n_sessions), np.nan)
for ni in range(n_neurons):
    for si in range(n_sessions):
        a = actual_max[ni, si]
        p = pred_max[ni, si]
        v = ~np.isnan(a) & ~np.isnan(p)
        if v.sum() >= 3 and np.std(a[v]) > 0 and np.std(p[v]) > 0:
            cv_state_corrs_max[ni, si] = stats.pearsonr(a[v], p[v])[0]
mean_state_corrs_max = np.nanmean(cv_state_corrs_max, axis=1)

mean_state_mean_arr = results_tc['mean_state_correlations']
valid = ~np.isnan(mean_state_mean_arr) & ~np.isnan(mean_state_corrs_max)
r_mean = mean_state_mean_arr[valid]
r_max  = mean_state_corrs_max[valid]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: max distribution
ax = axes[0]
ax.hist(r_max, bins=30, color='purple', alpha=0.7, edgecolor='black')
ax.axvline(0, color='black', linestyle='--', linewidth=2)
ax.axvline(r_max.mean(), color='red', linewidth=2, label=f'Mean = {r_max.mean():.3f}')
t, p = stats.ttest_1samp(r_max, 0)
ax.text(0.05, 0.95, f'n={len(r_max)}\nt={t:.2f}\np={p:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('State-MAX correlation')
ax.set_ylabel('Number of neurons')
ax.set_title('4-state MAX tuning corr')
ax.legend()

# Panel 2: per-neuron scatter mean vs max
ax = axes[1]
ax.scatter(r_mean, r_max, alpha=0.5, s=20, c='teal')
ax.plot([-1, 1], [-1, 1], 'k--', linewidth=1)
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_aspect('equal')
ax.set_xlabel('4-state MEAN corr')
ax.set_ylabel('4-state MAX corr')
ax.set_title('Per-neuron: mean vs max')

# Panel 3: per-neuron Delta
ax = axes[2]
diffs = r_max - r_mean
ax.hist(diffs, bins=30, color='darkorange', alpha=0.7, edgecolor='black')
ax.axvline(0, color='black', linestyle='--', linewidth=2)
ax.axvline(diffs.mean(), color='red', linewidth=2, label=f'Mean \u0394 = {diffs.mean():+.3f}')
t, p = stats.ttest_1samp(diffs, 0)
ax.text(0.05, 0.95, f'n={len(diffs)}\nt={t:.2f}\np={p:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('Per-neuron r_max - r_mean')
ax.set_ylabel('Number of neurons')
ax.set_title('\u0394 (max - mean) per neuron')
ax.legend()

plt.suptitle('4-state tuning correlation: mean vs max',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Cross-mouse summary

In [ ]:
from datetime import datetime

all_results, all_nz_corrs = run_and_summarise_all_mice(
    data_dic,
    config,
    valid_sessions_dic=valid_sessions_dic,
    num_examples=6,
    save_dir=f'../data/figures/elasticnet_summary_svg_smooth_{datetime.now().strftime("%Y%m%d_%H%M%S")}',
    plot_smooth_sigma=10
)

## Extra single-mouse plots

In [ ]:
# Default: automatically filters to state-tuned neurons

results = run_cross_validated_regression_tuning_curves(data_dic, list(data_dic.keys())[6], config)


plot_example_betas(results, config, num_examples=30)

# Plot population summary of betas
plot_population_beta_summary(results, config)

In [ ]:
# Show all neurons, all sessions, smoothed curves
plot_polar_tuning_curves(results, config, data_dic=data_dic, mouse_recday=mouse_recday, num_examples=9)

# Show only sequence neurons (peak lag >= 2)  
plot_polar_tuning_curves(results, config, data_dic=data_dic, mouse_recday=mouse_recday, 
                         num_examples=9, nonzero_lag_only=True)